# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("J3_Analytics_SQL_Gold").getOrCreate()
SILVER_MAIN_PATH = "/lakehouse/silver/orders_enriched"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").collect()[0][0])

## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [ ]:
df_dist_summary = spark.sql("""
SELECT
    COUNT(*) AS total_orders,
    AVG(delivery_days) AS mean_delivery_days,
    STDDEV(delivery_days) AS std_delivery_days,
    SKEWNESS(delivery_days) AS skewness_delivery_days,
    PERCENTILE_APPROX(delivery_days, 0.25) AS p25_delivery_days,
    PERCENTILE_APPROX(delivery_days, 0.50) AS median_delivery_days,
    PERCENTILE_APPROX(delivery_days, 0.75) AS p75_delivery_days,
    PERCENTILE_APPROX(delivery_days, 0.90) AS p90_delivery_days,
    PERCENTILE_APPROX(delivery_days, 0.95) AS p95_delivery_days,
    AVG(delay_days) AS mean_delay_days,
    AVG(is_late) AS late_rate
FROM v_silver
WHERE delivery_days IS NOT NULL AND delivery_days >= 0
""")
df_dist_summary.show(truncate=False)

**Justification statistique :** une skewness positive indique une queue vers les délais longs; si la moyenne dépasse la médiane, quelques livraisons très longues peuvent tirer la moyenne vers le haut. La médiane et l’IQR (Q3 − Q1) sont alors des indicateurs robustes. Vérifier le signe et les valeurs réelles dans le tableau calculé avant de conclure.

---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)

### Exercice 2.1 — Agrégation par segment/catégorie + Courbe cumulée de Pareto
Construisez en Spark SQL une première table Gold agrégée par une dimension métier pertinente de votre dataset incluant :
- Le volume d'observations, le total et les indicateurs robustes (**médiane**, **IQR**).
- Le rang via `DENSE_RANK() OVER (ORDER BY ... DESC)`.
- Le pourcentage cumulé du total (**analyse de Pareto 80/20**) via :
  `SUM(metric_total) OVER (ORDER BY metric_total DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(metric_total) OVER ()`.

Sauvegardez-la avec `.coalesce(1)` au format **Delta** dans `/lakehouse/gold/<nom_gold_1>`.

In [ ]:
GOLD_PARETO_PATH = "/lakehouse/gold/late_orders_by_state"

df_gold_1 = spark.sql("""
WITH state_counts AS (
    SELECT
        COALESCE(customer_state, 'UNKNOWN') AS customer_state,
        COUNT(*) AS total_orders,
        SUM(is_late) AS late_orders,
        AVG(delay_days) AS average_delay_days
    FROM v_silver
    WHERE order_delivered_customer_date IS NOT NULL
    GROUP BY COALESCE(customer_state, 'UNKNOWN')
),
ranked AS (
    SELECT *,
        SUM(total_orders) OVER (
            ORDER BY total_orders DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_orders,
        SUM(total_orders) OVER () AS all_orders
    FROM state_counts
)
SELECT *,
       cumulative_orders / all_orders AS cumulative_share
FROM ranked
ORDER BY total_orders DESC
""")

(df_gold_1.coalesce(1).write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").save(GOLD_PARETO_PATH))
df_gold_1.show(15, truncate=False)

## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [1]:
GOLD_TIME_PATH = "/lakehouse/gold/monthly_delivery"

df_gold_2 = spark.sql("""
WITH monthly AS (
    SELECT
        date_format(order_purchase_timestamp, 'yyyy-MM') AS purchase_month,
        COUNT(*) AS total_orders,
        AVG(delivery_days) AS average_delivery_days,
        AVG(delay_days) AS average_delay_days,
        AVG(is_late) AS late_rate
    FROM v_silver
    WHERE order_delivered_customer_date IS NOT NULL
    GROUP BY date_format(order_purchase_timestamp, 'yyyy-MM')
)
SELECT *,
       LAG(average_delivery_days) OVER (ORDER BY purchase_month) AS previous_month_delivery_days,
       average_delivery_days -
       LAG(average_delivery_days) OVER (ORDER BY purchase_month) AS change_vs_previous_month
FROM monthly
ORDER BY purchase_month
""")

(df_gold_2.coalesce(1).write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").save(GOLD_TIME_PATH))
df_gold_2.show(20, truncate=False)

NameError: name 'spark' is not defined

## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'` et expliquez pourquoi la multiplication de micro-fichiers pénalise le NameNode HDFS.

In [ ]:
GOLD_CATEGORY_PATH = "/lakehouse/gold/category_performance"

df_gold_3 = spark.sql("""
SELECT
    COALESCE(product_category, 'unknown') AS product_category,
    COUNT(*) AS total_orders,
    AVG(order_value) AS average_order_value,
    AVG(delivery_days) AS average_delivery_days,
    AVG(delay_days) AS average_delay_days,
    AVG(is_late) AS late_rate,
    AVG(review_score) AS average_review_score
FROM v_silver
WHERE order_delivered_customer_date IS NOT NULL
GROUP BY COALESCE(product_category, 'unknown')
ORDER BY total_orders DESC
""")

(df_gold_3.coalesce(1).write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").save(GOLD_CATEGORY_PATH))

# OPTIMIZE est une commande Delta disponible selon la version du moteur.
try:
    spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").show(truncate=False)
except Exception as exc:
    print("OPTIMIZE non disponible dans cette configuration :", str(exc)[:300])

df_gold_3.show(20, truncate=False)